# Single-Fact Learning-Update Audit: NSR vs NeuralLP

This notebook reconstructs the quantitative tables in `Interpre_Results.md` from the committed per-event JSON. The narrow question is: **after observing one fact, is the learned-state change confined to the relational structures whose graph evidence changed?**

The experiment uses 30 independently evaluated validation facts on each of NATIONS and KINSHIP. Every selected fact is absent from training and closes at least one two-hop path already present in the training graph.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

candidates = [
    Path.cwd() / "single_fact_locality_results.json",
    Path.cwd() / "NSR" / "Interpretability" / "single_fact_locality_results.json",
]
RESULTS_PATH = next((p.resolve() for p in candidates if p.exists()), None)
if RESULTS_PATH is None:
    raise FileNotFoundError("Run this notebook from the repository root or NSR/Interpretability")

with RESULTS_PATH.open(encoding="utf-8") as handle:
    results = json.load(handle)

print(f"Loaded: {RESULTS_PATH}")
print(f"Events per dataset: {results['num_events_per_dataset']}; seed: {results['seed']}")

## Metrics

For a new fact $e$, let $\Delta c_\rho(e)$ be the before/after score change of a two-hop rule $\rho$. The model-independent set $\mathcal R(e)$ contains the rules whose graph body support or consequent-hit count changes after adding $e$:

$$
L_{\mathrm{rule}}(e)=
\frac{\sum_{\rho\in\mathcal R(e)}\Delta c_\rho(e)^2}
     {\sum_\rho\Delta c_\rho(e)^2}.
$$

The concentration of the native internal update is measured by the inverse participation ratio:

$$
p_i=\frac{\Delta\theta_i^2}{\lVert\Delta\theta\rVert_2^2},\qquad
N_{\mathrm{eff}}=\frac{1}{\sum_i p_i^2},\qquad
R_{\mathrm{eff}}=\frac{N_{\mathrm{eff}}}{P}.
$$

Higher $L_{\mathrm{rule}}$ and lower $N_{\mathrm{eff}}/P$ indicate a more localized, easier-to-audit update. NSR coordinates are its two-hop C-layer connections; NeuralLP coordinates are all trainable parameters changed by one native Adam step.

In [ ]:
# Reconstruct the reviewer-facing aggregate table.
rows = []
for dataset in ("NATIONS", "KINSHIP"):
    for method in ("NSR", "NeuralLP"):
        summary = results["datasets"][dataset]["methods"][method]
        rows.append({
            "Dataset": dataset,
            "Method": method,
            "L_rule mean": summary["rule_update_locality"]["mean"],
            "L_rule std": summary["rule_update_locality"]["std"],
            "N_eff mean": summary["effective_updated_parameters"]["mean"],
            "N_eff std": summary["effective_updated_parameters"]["std"],
            "P": summary["parameter_count"][0],
            "N_eff/P (%)": 100 * summary["effective_update_ratio"]["mean"],
            "ratio std (%)": 100 * summary["effective_update_ratio"]["std"],
        })

aggregate_df = pd.DataFrame(rows)
aggregate_df.style.format({
    "L_rule mean": "{:.4f}",
    "L_rule std": "{:.4f}",
    "N_eff mean": "{:,.1f}",
    "N_eff std": "{:,.1f}",
    "P": "{:,}",
    "N_eff/P (%)": "{:.4f}",
    "ratio std (%)": "{:.4f}",
})

## Per-event inspection

Change `DATASET` below to inspect either dataset. Each row corresponds to one fact evaluated from the same base model state; updates do not accumulate.

In [ ]:
DATASET = "NATIONS"  # Change to "KINSHIP" and rerun this cell.

event_rows = []
for event in results["datasets"][DATASET]["events"]:
    for method, key in (("NSR", "nsr"), ("NeuralLP", "neural_lp")):
        record = event[key]
        event_rows.append({
            "event": event["event_index"],
            "fact": "(" + ", ".join(event["fact"]) + ")",
            "method": method,
            "closing paths": event["closing_path_count"],
            "relevant rules": event["relevant_rule_count"],
            "L_rule": record["rule_update_locality"],
            "N_eff": record["effective_updated_parameters"],
            "N_eff/P (%)": 100 * record["effective_update_ratio"],
        })

events_df = pd.DataFrame(event_rows)
events_df.head(10).style.format({
    "L_rule": "{:.4f}",
    "N_eff": "{:,.1f}",
    "N_eff/P (%)": "{:.4f}",
})

In [ ]:
# Distribution summary without relying on a figure in the rebuttal.
events_df.groupby("method")[["L_rule", "N_eff", "N_eff/P (%)"]].describe().round(4)

## Small rule-change audit

The saved case is the first eligible fact in the fixed seeded order, not a post-hoc selected example. `graph relevant` reports membership in the same graph-theoretic $\mathcal R(e)$ used for both methods.

In [ ]:
case = results["datasets"][DATASET]["case_event"]
case_rows = []
for method in ("NSR", "NeuralLP"):
    for rank, change in enumerate(case[method], start=1):
        case_rows.append({
            "method": method,
            "rank": rank,
            "rule": change["rule"],
            "before": change["before"],
            "after": change["after"],
            "delta": change["delta"],
            "graph relevant": change["graph_relevant"],
        })

print("Observed fact:", tuple(case["fact"]))
print("Closing paths:", case["closing_path_count"])
print("Graph-relevant rules:", case["relevant_rule_count"])
pd.DataFrame(case_rows).style.format({
    "before": "{:.6f}",
    "after": "{:.6f}",
    "delta": "{:+.6f}",
})

## Reproduction and interpretation boundary

The optional cell below reruns all 60 event audits and overwrites the raw result JSON. It may take several minutes and requires the released NeuralLP checkpoints plus the `nsr_tf1` Conda environment.

The experiment quantifies **learning-update locality**, not all possible meanings of interpretability. NSR uses exact sufficient-statistic incorporation, while NeuralLP receives one native optimizer step. The comparison therefore measures how localized each method's native response to the same fact is; it does not equate their learning rates.

In [ ]:
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    project_root = RESULTS_PATH.parents[2]
    command = [
        sys.executable,
        str(project_root / "NSR" / "Interpretability" / "evaluate_single_fact_locality.py"),
        "--datasets", "NATIONS", "KINSHIP",
        "--num-events", "30",
        "--seed", "42",
    ]
    subprocess.run(command, cwd=project_root, check=True)
else:
    print("Set RUN_FULL_EXPERIMENT=True to regenerate the raw results.")